# 03 - Inference

This notebook tests the predictions of the champion model, that is, the
version of `heart-disease-classifier` with the alias `champion` in the MLflow
model registry. The model is read from `models/champion/`, the folder to which
the champion is exported, so the MLflow database is not needed. The loading and
prediction functions are the ones used by the API (`api/inference.py`).

The champion has to be registered and exported before this notebook is run,
in one of two ways.

**Through the pipeline (usual way).** From the project root:

```bash
python -m pipeline.run_all
```

Step 5 registers the selected model and makes it the champion if it passes
the two gates; step 6 exports the champion to `models/champion/`.

**Manually, in the MLflow UI.** After notebook 02 has been run:

1. Start the UI: `mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000`
   and open http://localhost:5000.
2. In the experiment "Heart Disease Prediction", open the parent run with the
   tag `selected_model = true` (the model selected and tested in notebook 02).
3. Register its model under the name `heart-disease-classifier`.
4. Open the new version in the model registry and add the alias `champion`.
5. Export the champion: `python -m pipeline.step6_export_model`.

1. Loading of the champion model
2. Prediction for the patients of the test set
3. A patient with a missing value
4. Prediction through the API

In [ ]:
import json
import sys
import urllib.error
import urllib.request
from pathlib import Path

import pandas as pd

# project root, one level above notebooks/
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from api.inference import LABELS, MODEL_DIR, load_model, predict
from src.data.preparation import split_data
from src.models.evaluation import compute_metrics

pd.set_option("display.width", 200)

## 1. Loading of the champion model

The exported folder is checked first. If it is missing, the notebook stops
here with a message.

In [ ]:
if not (MODEL_DIR / "MLmodel").exists():
    print(f"No exported champion model was found in {MODEL_DIR}.")
    print("Register and export the champion first, as described at the top of this notebook.")
    raise SystemExit("Notebook stopped: the exported champion model is missing.")

print(f"Exported champion model found in {MODEL_DIR}")

The name and version are those of the registered model. The run ID and the
training time are those of the MLflow run that trained it.

In [ ]:
champion = load_model()

pd.Series({
    "name": champion.name,
    "version": champion.version,
    "run ID": champion.run_id,
    "trained at": champion.trained_at,
}).to_frame("champion model")

## 2. Prediction for the patients of the test set

The test set is the same 20% of the patients that was kept aside in training
(same split, same seed). The model has not seen these patients.

In [ ]:
X_train, X_test, y_train, y_test = split_data()

# predict() takes one dict per patient, with None for a missing value
patients = X_test.astype(object).where(X_test.notna(), None).to_dict(orient="records")

results = pd.DataFrame(predict(champion.model, patients), index=X_test.index)
results.insert(0, "true outcome", y_test.map(LABELS))
results.head()

Metrics on all the patients of the test set:

In [ ]:
probability = champion.model.predict_proba(X_test)[:, 1]

pd.Series(compute_metrics(y_test, probability)).round(3).to_frame("test set")

The metrics are the same as those of the selected model in notebook 02
(section 11). So the exported model is the model that was trained and tested.

## 3. A patient with a missing value

The pipeline fills a missing value in the same way as in training: the median
for a numerical feature and the most frequent value for a categorical feature.
The API accepts `null` for a feature, and reports the filled features in
`filled_fields`.

The sample request has `thal` = 7 (reversible defect). Here `thal` is left
out, and the pipeline fills it with 3 (normal), the most frequent value.

In [ ]:
sample = json.loads((ROOT / "api" / "sample_request.json").read_text())
sample_missing_thal = dict(sample, thal=None)

pd.DataFrame(
    predict(champion.model, [sample, sample_missing_thal]),
    index=["sample request", "sample request, thal missing"],
)

## 4. Prediction through the API

The sample request is sent to the `/predict` endpoint of the API deployed on
Minikube, and `/model-info` gives the model that the API serves. This section
needs the API to be running on Minikube and its port forwarded to this
computer, in a separate terminal:

```bash
kubectl port-forward svc/heart-disease-api 8000:80
```

In [ ]:
API_URL = "http://localhost:8000"


def call_api(path, body=None):
    """Send a GET request, or a POST request when a body is given, and
    return the response as a dict."""
    data = None if body is None else json.dumps(body).encode()
    request = urllib.request.Request(
        API_URL + path, data=data, headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(request, timeout=10) as response:
        return json.load(response)


try:
    model_info = call_api("/model-info")
    api_result = call_api("/predict", sample)
except (urllib.error.URLError, TimeoutError) as error:
    print(f"The API at {API_URL} could not be reached: {error}")
    print("Check that the API is running on Minikube and that the port-forward is started.")
else:
    local_result = predict(champion.model, [sample])[0]
    comparison = pd.DataFrame({
        "this notebook": {
            "model version": champion.version,
            "run ID": champion.run_id,
            "label": local_result["label"],
            "confidence": local_result["confidence"],
        },
        "API": {
            "model version": model_info["model_version"],
            "run ID": model_info["run_id"],
            "label": api_result["label"],
            "confidence": api_result["confidence"],
        },
    })
    display(comparison)

The Docker image of the API is built from the same `models/champion/` folder
that this notebook reads, so both show the same run ID and give the same
prediction and confidence.

## Summary

- The exported model is loaded without the MLflow database and gives the same
  test metrics as in notebook 02.
- A missing value is filled by the pipeline and reported in `filled_fields`.
- The API on Minikube gives the same prediction for the same patient.